# Pipeline CLIP + Graph trên Kaggle: cấp độ 1 trên một dataset, xAI cấp độ 3 trên Visual Genome

Chọn dataset bằng biến `DATASET` ở ô cấu hình:

| `DATASET` | Notebook làm gì |
|---|---|
| `visual_genome` | **xAI cấp độ 3** trên mô hình đã báo cáo `clip_graph_vg_lr1e4` (GraphSAGE một lớp trong `results/tables/main.csv`, ADR-0046; 3 seed, `alpha` chọn trên val). Đánh giá giải thích bằng fidelity, validity, sparsity trên ≥ 10 truy vấn (mục 2–4) |
| `flickr30k` | **Tập thứ hai của cấp độ 2** (ADR-0050): chạy lại pipeline cấp độ 1, chỉ đổi dataset. Dựng đồ thị, train 3 mô hình (`clip_graph_flickr30k_lr1e4`, `clip_nograph_flickr30k`, `clip_graph_flickr30k_lr1e4_shuffled`), giải thích cơ bản 5 thành công + 5 thất bại trên val (mục 1d), test một lần ở mục 4b. Bỏ qua xAI. Input cần thêm: embedding Flickr30k (`.../flickr30k/ViT-B-32__laion2b_s34b_b79k/meta.json`), cùng một trong hai thứ: đồ thị VG `contains_clip_top16_v1` (nguồn từ vựng khái niệm) hoặc đồ thị Flickr30k đã dựng |

Notebook này chỉ gọi các script có sẵn trong repo:

| Bước | Làm gì | Script |
|---|---|---|
| 1 | Lấy embedding, đồ thị, checkpoint từ **Kaggle input**. Thiếu thì tự dựng hoặc tự train | `build_graph.py` (VG), `build_graph_from_vocabulary.py` + `shuffle_graph.py` (Flickr30k), `train_graph_rerank.py` |
| 1d | **Giải thích cơ bản** (cấp độ 1): 5 thành công + 5 thất bại trên val, Occlusion + kiểm tra xoá cạnh | `explain.py` |
| 2 (VG) | Chạy đánh giá giải thích trên **val**: 3 explainer × `m` × 20 truy vấn × 3 seed | `evaluate_explanations.py` |
| 3 (VG) | Xem bảng tổng hợp và vài bản ghi cụ thể | đọc `summary.md`, `evaluation.json` |
| 4 (VG) | **Test, đúng một lần**, với explainer và `m` mà nhóm đã chốt và commit | `evaluate_explanations.py` |
| 4b (Flickr30k) | **Test cấp độ 1, đúng một lần**: CLIP thuần + 3 mô hình, mỗi seed, rồi tổng hợp mean ± std | `evaluate_clip.py`, `evaluate_graph_rerank.py`, `summarize_runs.py` |

Quy trình chi tiết: `docs/research/xai-method-proposal.md` mục 7d.

## Chuẩn bị trên Kaggle

1. **Settings:** bật **Internet**. CPU là đủ; GPU không bắt buộc.
2. **Add-ons → Secrets:** thêm `GITHUB_TOKEN` (repo là private). Dùng token classic, chỉ tick quyền `repo`, đặt hạn ngắn. Không dán token vào notebook.
3. **Add Input** (notebook tự tìm theo tên thư mục, đặt ở đâu trong input cũng được):

| Cần | Tìm theo đường dẫn | Bắt buộc? |
|---|---|---|
| Embedding CLIP Visual Genome | `.../visual_genome/ViT-B-32__laion2b_s34b_b79k/meta.json` | **Có** |
| Đồ thị CLIP top-16 | `.../contains_clip_top16_v1/graph.npz` | Không. Thiếu thì dựng lại, khi đó cần `objects.json` |
| Checkpoint đã train | `.../clip_graph_vg_lr1e4/seed<k>/model.pt` + `selection.json` | Không. Seed nào thiếu thì train lại (khoảng 30–60 phút một seed trên CPU) |
| `objects.json` (Visual Genome v1.4) | `.../objects.json` | Không. Thiếu cả đồ thị lẫn file này thì notebook tự tải từ trang Visual Genome (cần bật Internet) và kiểm sha256 |

Nên dùng **đồ thị và checkpoint của người đã train** (bản đã cho các số trong `main.csv`). Nếu notebook phải train lại, kết quả là của **mô hình train lại**, và báo cáo phải ghi rõ điều đó.

4. **Save Version → Save & Run All.** Mọi output nằm trong `/kaggle/working/graph-semantic-image-editing/results/runs/`.

## 0. Cấu hình

In [ ]:
REPO_URL = "https://github.com/FluteInTheRain/graph-semantic-image-editing.git"
BRANCH = "feat/flickr30k-level1"
# "visual_genome": xAI level 3 on the reported model.
# "flickr30k": level-2 second dataset, the level-1 pipeline on new data (ADR-0050).
DATASET = "flickr30k"
EXPLAIN_CONFIG = "configs/explain_vg.yaml"
SEEDS = [0, 1, 2]
THREADS = 4  # Kaggle CPU sessions have 4 cores

SETTINGS = {
    "visual_genome": {
        "features": "artifacts/features/visual_genome/ViT-B-32__laion2b_s34b_b79k",
        "graph": "artifacts/graphs/visual_genome/contains_clip_top16_v1",
        "clip_config": "configs/clip_vg.yaml",
        "text_file": "region_descriptions.json",  # optional: query text in 1d
        "ranking_configs": ["configs/clip_graph_vg_lr1e4.yaml"],  # the reported model (main.csv)
    },
    "flickr30k": {
        "features": "artifacts/features/flickr30k/ViT-B-32__laion2b_s34b_b79k",
        "graph": "artifacts/graphs/flickr30k/contains_clip_top16_v1",
        "clip_config": "configs/clip_flickr30k.yaml",
        "text_file": "dataset_flickr30k.json",
        "ranking_configs": [
            "configs/clip_graph_flickr30k_lr1e4.yaml",
            "configs/clip_nograph_flickr30k.yaml",
            "configs/clip_graph_flickr30k_lr1e4_shuffled.yaml",
        ],
    },
}
FEATURES = SETTINGS[DATASET]["features"]
GRAPH = SETTINGS[DATASET]["graph"]
CLIP_CONFIG = SETTINGS[DATASET]["clip_config"]
RANKING_CONFIGS = SETTINGS[DATASET]["ranking_configs"]
RUN_XAI = DATASET == "visual_genome"
# Level-1 basic explanation (5 success + 5 failure val queries) of the first ranking config.
RUN_BASIC_EXPLAIN = True

# Test runs ONCE (ADR-0009). xAI: only after main_explainer and top_m are committed.
# Level 1 on flickr30k: only after the val runs below look complete.
RUN_TEST = False
RUN_LEVEL1_TEST = False

## 1. Môi trường, code và dữ liệu

In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()


def github_token():
    """The GITHUB_TOKEN secret of Kaggle, or None. Never printed or saved."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def clone_repo(branch, url, folder):
    """Clone the private repo with the token, then drop the token from the remote URL."""
    token = github_token()
    if token is None:
        raise SystemExit("Add a GitHub token as the Kaggle secret GITHUB_TOKEN, then rerun.")
    authed_url = url.replace("https://", f"https://x-access-token:{token}@")
    # A list (no shell) so the token is not echoed; no prompt, so a bad token fails at once.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    result = subprocess.run(
        ["git", "clone", "-q", "-b", branch, authed_url, folder],
        env=env,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise SystemExit("git clone failed: " + result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", folder, "remote", "set-url", "origin", url], check=True)


if Path("../src/mmxai").exists():  # opened from notebooks/ in a local clone
    os.chdir("..")
elif not Path("src/mmxai").exists():  # Kaggle: fetch the code
    if not Path("graph-semantic-image-editing").exists():
        clone_repo(BRANCH, REPO_URL, "graph-semantic-image-editing")
    os.chdir("graph-semantic-image-editing")
    # torch is preinstalled on Kaggle: add only what is missing.
    !pip install -q -e . torch-geometric faiss-cpu open-clip-torch
# `pip install -e .` is only seen by new processes (the scripts below); this kernel
# gets src/ on its path directly.
if str(Path.cwd() / "src") not in sys.path:
    sys.path.insert(0, str(Path.cwd() / "src"))
print("repo:", Path.cwd(), "| commit:", subprocess.getoutput("git rev-parse --short HEAD"))

In [ ]:
import yaml


def link_from_input(pattern, target):
    """Symlink the first /kaggle/input match of pattern to target; True if target exists."""
    target = Path(target)
    if not target.exists() and ON_KAGGLE:
        # A few levels only: "**" would walk every file of a large input (e.g. VG images).
        found = []
        for depth in range(6):
            found += sorted(glob.glob("/kaggle/input/" + "*/" * depth + pattern))
        if found:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.symlink_to(Path(found[0]))
    return target.exists()


def model_dir(config):
    with open(config, encoding="utf-8") as file:
        return yaml.safe_load(file)["model_dir"]


assert link_from_input(f"{DATASET}/ViT-B-32__laion2b_s34b_b79k", FEATURES), (
    f"No CLIP features: add the {DATASET} embedding output as a Kaggle input"
)
if DATASET == "visual_genome":
    has_graph = link_from_input("contains_clip_top16_v1", GRAPH)
else:
    has_graph = link_from_input(f"{DATASET}/contains_clip_top16_v1", GRAPH)
has_objects = link_from_input("objects.json", "data/visual_genome/objects.json")
# Link seed by seed: a missing seed is trained into a writable folder below.
missing_seeds = {}
for config in RANKING_CONFIGS:
    folder = model_dir(config)
    name = Path(folder).name
    missing_seeds[config] = [
        seed for seed in SEEDS if not link_from_input(f"{name}/seed{seed}", f"{folder}/seed{seed}")
    ]
print("graph from input:", has_graph, "| objects.json:", has_objects)
print("seeds to train:", missing_seeds)

## 1b. Dựng đồ thị nếu input không có

- **Visual Genome:** đồ thị đã báo cáo có `graph.npz` sha256 `33fb4c6b…` (`results/records/20261113_cap-do-1/README.md` mục 3). Dựng lại mà ra hash khác thì đồ thị đã khác, và phải ghi rõ trong báo cáo.
- **Flickr30k:** cần đồ thị VG làm nguồn từ vựng khái niệm (ADR-0050). Notebook dựng thêm bản xáo cạnh cho ablation.

In [ ]:
from mmxai.utils.runs import file_sha256

REPORTED_GRAPH_SHA256 = "33fb4c6b95eaed9ad34ed56b423006b068840f969742d91d4b2fef7a5edc7c48"
VG_GRAPH = SETTINGS["visual_genome"]["graph"]

# objects.json v1.4 (docs/guides/xay-do-thi.md): the concept vocabulary is counted on train.
OBJECTS_URL = "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/objects.json.zip"
OBJECTS_SHA256 = "94238b6187c0ea1f97acc1fdaec106bd3dee12bb0f81b23dc8b07e9173514985"
OBJECTS = Path("data/visual_genome/objects.json")

if DATASET == "visual_genome" and not has_graph:
    if not has_objects:  # not in the input: download it (needs Internet on)
        OBJECTS.parent.mkdir(parents=True, exist_ok=True)
        !curl -L --fail -s -o {OBJECTS}.zip {OBJECTS_URL}
        !unzip -o -q {OBJECTS}.zip -d {OBJECTS.parent} && rm {OBJECTS}.zip
    # The same file as the reported graph was built from, or stop.
    assert file_sha256(OBJECTS) == OBJECTS_SHA256, f"{OBJECTS} is not objects.json v1.4"
    !python scripts/build_graph.py --config configs/graph_vg_clip_topk.yaml

if DATASET == "flickr30k":
    if not has_graph:
        assert link_from_input("contains_clip_top16_v1", VG_GRAPH), (
            "Add the VG graph contains_clip_top16_v1 as an input: it holds the concept vocabulary"
        )
        print("VG vocabulary graph sha256:", file_sha256(f"{VG_GRAPH}/graph.npz"))
        graph_config = "configs/graph_flickr30k_clip_topk.yaml"
        !python scripts/build_graph_from_vocabulary.py --config {graph_config}
    if not Path("artifacts/graphs/flickr30k/contains_clip_top16_shuffled_seed0").exists():
        !python scripts/shuffle_graph.py --config configs/graph_flickr30k_clip_topk_shuffled.yaml

graph_sha = file_sha256(f"{GRAPH}/graph.npz")
print("graph.npz sha256:", graph_sha)
if DATASET == "visual_genome":
    print("same graph as the reported model:", graph_sha == REPORTED_GRAPH_SHA256)

## 1c. Train các seed còn thiếu, rồi chấm CLIP thuần trên val

Train chỉ học trên train; epoch và `alpha` chọn trên val (ADR-0047). Mỗi config dùng đúng script `train_graph_rerank.py`.

In [ ]:
for config, seeds in missing_seeds.items():
    if seeds:
        train_args = f"--config {config} --seeds {' '.join(str(seed) for seed in seeds)}"
        !python scripts/train_graph_rerank.py {train_args} --threads {THREADS}
    else:
        print(config, ": every seed comes from the input")
!python scripts/evaluate_clip.py --config {CLIP_CONFIG} --seed 0

## 1d. Giải thích cơ bản của cấp độ 1: 5 thành công + 5 thất bại trên val

Đề yêu cầu cho mọi cấp độ: hiển thị quan hệ dùng khi xếp hạng, và phân tích 5 truy vấn thành công + 5 thất bại. `scripts/explain.py` làm việc đó trên **chính mô hình đã xếp hạng** (config đầu tiên trong `RANKING_CONFIGS`, seed 0). Cụ thể:
- dùng Occlusion để tìm các cạnh khái niệm → ảnh đóng góp nhiều nhất vào điểm đồ thị;
- bỏ 3 cạnh đó rồi xem điểm và hạng thay đổi ra sao (kiểm tra xoá cạnh).

Chỉ chạy trên val (ADR-0009). Câu truy vấn được đọc từ file thô nếu có trong input (`dataset_flickr30k.json` hoặc `region_descriptions.json`). Không có file thì báo cáo hiện id thay cho câu.

In [ ]:
from IPython.display import Markdown, display

TEXT_FILE = SETTINGS[DATASET]["text_file"]
TEXT_DIR = Path("data") / DATASET
has_text = link_from_input(TEXT_FILE, TEXT_DIR / TEXT_FILE)
if RUN_BASIC_EXPLAIN:
    explain_args = f"--config {RANKING_CONFIGS[0]} --seed 0 --split val --num-edges 3"
    if has_text:
        explain_args += f" --text-data-dir {TEXT_DIR}"
    !python scripts/explain.py {explain_args} --threads {THREADS}
    EXPLAIN_RUN = sorted(Path("results/runs").glob("*_explain_seed0"))[-1]
    display(Markdown((EXPLAIN_RUN / "analysis.md").read_text()))
else:
    print("RUN_BASIC_EXPLAIN is False: basic explanation not run")

## 2. Đánh giá giải thích trên val

Script tự dừng nếu MRR val tính lại khác số đã lưu lúc train. Như vậy, phần giải thích chắc chắn thuộc về đúng mô hình và đúng `alpha` đã xếp hạng.

Với mỗi seed và mỗi truy vấn trong 20 truy vấn bốc ngẫu nhiên một lần, script:
- giải thích ảnh đứng hạng 1 bằng Occlusion, Integrated Gradients và GNNExplainer;
- lấy top `m` cạnh làm giải thích;
- đo Fid+/Fid−, mức tụt hạng, robust fidelity, so với xoá cạnh ngẫu nhiên, sparsity và validity.

In [ ]:
if RUN_XAI:
    !python scripts/evaluate_explanations.py --config {EXPLAIN_CONFIG} --threads {THREADS}
else:
    print("xAI is run on visual_genome only")

## 3. Kết quả val

In [ ]:
import json

from IPython.display import Markdown, display

if RUN_XAI:
    VAL_RUN = sorted(Path("results/runs").glob("*_explain_vg_val"))[-1]
    display(Markdown((VAL_RUN / "summary.md").read_text()))
    print("files:", VAL_RUN)

**Cách đọc bảng:**
- **Fid+** cao hơn **random Fid+**, và **p (Fid+ > random)** < 0.05: bỏ cạnh trong giải thích làm điểm giảm nhiều hơn bỏ cùng số cạnh ngẫu nhiên. Đây là bằng chứng giải thích trung thực.
- **rank drop**: bỏ các cạnh trong giải thích thì ảnh tụt bao nhiêu hạng (dương nghĩa là tụt).
- **Fid−** gần 0: chỉ giữ các cạnh trong giải thích thì điểm vẫn gần như giữ nguyên, tức giải thích là "đủ".
- **sparsity** cao: giải thích gọn. **validity** (`all_in_reach_share`, `connected_share` trong `evaluation.json`) phải bằng 1.

Bảng dưới là một vài bản ghi của explainer chính được đề xuất, làm bằng chứng cụ thể theo từng truy vấn.

In [ ]:
import pandas as pd

if RUN_XAI:
    evaluation = json.loads((VAL_RUN / "evaluation.json").read_text())
    proposal = evaluation["summary"]["val_proposal"]
    print("val proposal:", proposal)
    records = pd.DataFrame(evaluation["records"])
    chosen = records[
        (records["explainer"] == proposal["main_explainer"])
        & (records["top_m"] == proposal["top_m"])
    ]
    columns = [
        "seed",
        "query_row",
        "image_id",
        "concepts",
        "fidelity_plus",
        "random_plus_mean",
        "rank_drop",
        "fidelity_minus",
        "sparsity",
        "connected_share",
    ]
    display(chosen[chosen["seed"] == SEEDS[0]][columns])

## 4. Test: chạy đúng một lần

Trước khi bật `RUN_TEST`:
1. Nhóm đọc bảng val và chốt `main_explainer` và `top_m`.
2. Ghi hai giá trị đó vào `configs/explain_vg.yaml`, **commit và push** lên nhánh.
3. Chạy lại notebook với `RUN_TEST = True`.

Ô dưới đây đọc config **đã commit** và chỉ đổi `eval_split` thành `test`. Nếu config chưa có lựa chọn thì script từ chối chạy.

⚠️ Script chặn lần chạy test thứ hai **trong cùng thư mục `results/runs/`**. Mỗi phiên Kaggle bắt đầu với một thư mục trống, nên script không thấy được các lần chạy trước. Sau khi chạy test, hãy chép thư mục kết quả vào `results/records/` rồi commit, và **không chạy lại**.

In [ ]:
if RUN_TEST and RUN_XAI:
    with open(EXPLAIN_CONFIG, encoding="utf-8") as file:
        values = yaml.safe_load(file)
    values["eval_split"] = "test"
    # A copy outside configs/: the committed file stays the val config.
    test_config = Path("explain_vg_test.yaml").resolve()
    test_config.write_text(yaml.safe_dump(values, sort_keys=False), encoding="utf-8")
    !python scripts/evaluate_explanations.py --config {test_config} --threads {THREADS}
    TEST_RUN = sorted(Path("results/runs").glob("*_explain_vg_test"))[-1]
    display(Markdown((TEST_RUN / "summary.md").read_text()))
else:
    print("RUN_TEST is False: test not run")

## 4b. Cấp độ 1 trên Flickr30k: test đúng một lần

Bật `RUN_LEVEL1_TEST` chỉ sau khi đã xem các run val ở trên. Ô này **không chọn gì**: nó chấm test bằng `alpha` và checkpoint đã chọn trên val, mỗi seed một lần (ADR-0009, ADR-0047). Sau đó tổng hợp mean ± std ra `level1_flickr30k_test.csv`.

⚠️ Giống mục 4: chép thư mục `results/runs/*_flickr30k*` về `results/records/` rồi commit, và **không chạy lại**.

In [ ]:
if RUN_LEVEL1_TEST:
    assert DATASET == "flickr30k", "level-1 test of visual_genome was already run once"
    for seed in SEEDS:  # noqa: B007 (used by the ! lines)
        !python scripts/evaluate_clip.py --config {CLIP_CONFIG} --seed {seed} --split test
        for config in RANKING_CONFIGS:  # noqa: B007 (used by the ! line)
            !python scripts/evaluate_graph_rerank.py --config {config} --seed {seed}
    !python scripts/summarize_runs.py --split test --output level1_flickr30k_test.csv
    !python scripts/summarize_runs.py --split val --output level1_flickr30k_val.csv
    display(pd.read_csv("level1_flickr30k_test.csv"))
else:
    print("RUN_LEVEL1_TEST is False: level-1 test not run")

## 5. Lấy kết quả về

Kaggle giữ mọi thứ trong `/kaggle/working/`. Các file cần lấy:
- `graph-semantic-image-editing/results/runs/*_explain_vg_val/` (và `*_explain_vg_test/` nếu có): mỗi thư mục gồm `evaluation.json`, `summary.md`, `config.yaml`, `env.json`.
- Flickr30k: `results/runs/*_flickr30k*` (gồm `*_explain_seed0/analysis.md`), `level1_flickr30k_test.csv`, `level1_flickr30k_val.csv` và `artifacts/models/flickr30k/`.
- Nếu đã train lại: `graph-semantic-image-editing/artifacts/models/visual_genome/clip_graph_vg_lr1e4/`, cùng các thư mục run `*_clip_graph_vg_lr1e4_seed<k>`.

Số liệu đưa vào báo cáo phải truy ngược được về đúng các thư mục này (ADR-0010).

In [ ]:
for run in sorted(Path("results/runs").iterdir()):
    print(run)